## Reviewer 3 (e9Ho) -- "significant test for headline table"

Weakness 7: *"Tables 1-2 report no significance test (only the spatial Fig.3 does), and
some 'wins' overlap in std (e.g. Immune 0.62+-0.08 vs 0.55+-0.04)."*

"Headline table" = Table 1 in the paper (modularity, batch entropy, purity -- scProto vs.
SEACells/MetaQ/scPoli(cVAE)/Parametric UMAP/Harmony/scVI). This notebook is **read-only**:
every baseline needed for Table 1 and the rare-cell Table 2 is already cached on disk for
all three RNA-seq datasets (pancreas, lung, pbmc-immune) -- verified directly against the
Drive folders, not assumed. Nothing here trains or computes anything new; it:

1. **Rebuilds Table 1 and the rare-cell Table 2** from what's already on disk.
2. **Runs the significance tests** already implemented in `paper_figures.py`:
   - `graph_batch_significance` -- one-sided Mann-Whitney U (scProto > baseline),
     Bonferroni-corrected, on Table 1's raw per-batch/per-metacell arrays
     (`modularity_per_batch.csv`, `purity_per_mc.csv`, `batch_entropy_per_mc.csv`).
   - `graph_batch_significance_paired` -- the recommended version for modularity: paired
     one-sided Wilcoxon signed-rank on `modularity_per_batch.csv` (batch is a shared,
     method-independent unit across every method on a dataset, so pairing removes
     shared batch-to-batch noise and is more powerful than the unpaired test).
   - `rare_metric_significance_paired` -- same idea for Table 2's rare-cell F1/homogeneity.

Run top to bottom in Colab.


## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Install cell -- same pattern as batch_correct_then_cluster_baselines.ipynb.
# Only needed the first time on a fresh runtime; skip/comment out if these are already
# installed in this session. RESTART THE RUNTIME after this cell before running below --
# numpy/scipy/anndata are C-extension linked, an in-process upgrade won't reliably take
# effect on already-imported modules.
!pip install -q scarches faiss-gpu-cu12 scib-metrics
!pip install git+https://github.com/dpeerlab/SEACells.git --quiet --no-deps
!pip install numpy scipy --upgrade -q
!pip install -q palantir harmonypy
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
!pip install --upgrade --force-reinstall numpy cupy-cuda12x
!pip install "numpy<2.3"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 114.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 91.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.1
    Uninstalling numpy-2.5.1:
      Successfully uninstalled numpy-2.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
seacells 0.3.3 requires pyranges, which is not installed.
anndata 0.13.2 requires scipy!=1.17.0,>=1.14, but you have scipy 1.13.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
access 1.1.10.post3 requires scipy>=1.14.1, but you have scipy 1.13.1 which is incompatible.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.66.0 which is incompatible.
tsfresh 0.21.2 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 

In [23]:
%run /content/drive/MyDrive/codes/interpretable-prototype/notebooks/nb_setup.py


nb_setup done. Available: get_trainer, run_mc_task, fig_*, LAMBDA_PROTO_UMAP, LAMBDA_PROTO_UMAP_PRECON, LAMBDA_PARAM_UMAP, LAMBDA_RECON_ONLY, train_sure, eval_sure_task1/2/3
Configs: {'LAMBDA_PROTO_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'proto'}, 'LAMBDA_PARAM_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'embedding'}, 'LAMBDA_RECON_ONLY': {'lambda_umap': 0, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 1, 'lambda_proto_recon': 0.0}}


In [24]:
from interpretable_ssl.datasets.dataset_configs import DATASETS
from interpretable_ssl.configs.paths import get_dataset_model_dir, get_metaq_model_dir
from interpretable_ssl.evaluation.metric_helpers.result_tables import extract_model_key
from interpretable_ssl.evaluation.paper_figures import (
    graph_batch_significance, graph_batch_significance_paired,
    rare_metric_significance_paired,
)

print("extra imports ready")


extra imports ready


## Config

`stage1z` (= scPoli/Stage-1 cVAE latent -> Leiden, the paper's "scPoli (cVAE)" baseline),
`harmony`, and `scvi_gauss` (Gaussian-reconstruction scVI -- the intended comparison,
loss-matched closer to scProto's own MSE Stage-1 pretrain; plain ZINB scVI is not used)
are the three correction methods compared here. `scvi_gauss` may need a local Drive sync
to show up if it was just trained.


In [25]:
RNA_SEQ_DATASETS = ['pancreas', 'lung', 'pbmc-immune']
dataset_display_names = {'pancreas': 'Pancreas', 'lung': 'Lung', 'pbmc-immune': 'Immune'}

# 'scvi_gauss' (Gaussian reconstruction, loss-matched closer to scProto's own MSE
# Stage-1 pretrain) is the intended scVI comparison, not plain ZINB 'scvi' -- see the
# MODEL_KEYWORDS cell below for why an exact-match keyword is required here (a plain
# 'scvi' substring keyword ambiguously matches both folders and silently picks
# whichever sorts last alphabetically).
CORRECTION_METHODS = ['stage1z', 'harmony', 'scvi_gauss']
METHOD_DISPLAY_NAMES = {
    'stage1z':    'scPoli (cVAE)',
    'harmony':    'Harmony',
    'scvi_gauss': 'scVI (Gaussian)',
}


## Load already-cached baselines (read-only -- no new compute)

`stage1z` (scPoli/cVAE) and `harmony` have complete `seacell_X_{method}` /
`leiden_X_{method}_K{K}` runs on disk for all three RNA-seq datasets. `scvi_gauss` should
too (trained separately) -- if the MODEL_KEYWORDS/significance cells below can't find it
for a dataset, it's most likely a Drive sync lag on this machine, not a missing run;
re-check after a sync rather than retraining.


## MetaQ (read-only check, no training fallback)

MetaQ's metrics live under `MODEL_DIR/<ds_id>/metaq/` (`get_metaq_model_dir`), which is a
subdirectory of the same per-dataset model dir the other baselines use -- so once its
`metrics.json`/CSVs are written, it's discoverable by the same `keyword='metaq'`
substring match the significance functions use below, no special-casing needed.

Already verified present for all three datasets, so this is just a sanity check --
it does **not** fall back to retraining (there's no saved MetaQ model checkpoint to
recover from anyway; a from-scratch MetaQ run is real GPU compute we don't need here).


In [26]:
import os

for _ds in RNA_SEQ_DATASETS:
    _metrics_path = os.path.join(get_metaq_model_dir(_ds), 'metrics.json')
    status = "OK (cached)" if os.path.exists(_metrics_path) else "MISSING"
    print(f"[{_ds}] MetaQ metrics.json: {status}")


[pancreas] MetaQ metrics.json: OK (cached)
[lung] MetaQ metrics.json: OK (cached)
[pbmc-immune] MetaQ metrics.json: OK (cached)


## Build the shared method -> display-name map

Mirrors the verified `MODEL_KEYWORDS` construction in
`batch_correct_then_cluster_baselines.ipynb` (scProto's canonical runs there were
cross-checked against the paper's published numbers -- bit-identical on multiple metrics
per dataset). `'metaq'` stays a substring keyword (its folder name has no dataset-specific
suffix, so it's unambiguous). `param_umap` and `scvi_gauss` are pinned by **exact** folder
name instead -- a plain substring keyword for either one silently matched the wrong run
(cvae_e100 instead of cvae_e50 for Parametric UMAP; plain ZINB scVI instead of Gaussian)
in different cells of the same notebook run. See the comments in the next cell for why.


In [27]:
# scProto's canonical runs -- verified against the paper's published Table 1/2 numbers
# (see batch_correct_then_cluster_baselines.ipynb, 'scProtoCanonicalCell01' for the
# original verification). All three normalize to one shared stripped key via
# extract_model_key(), so ONE row ('scProto') covers all three datasets. All three use
# cvae_e50 (50-epoch Stage-1 pretrain) -- Parametric UMAP below is pinned to match.
SCPROTO_CANONICAL_RUNS = {
    'proto_umap_ds-panc_NP220_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31',
    'proto_umap_ds-lung_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31',
    'proto_umap_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31',
}
SCPROTO_KEY = extract_model_key(next(iter(SCPROTO_CANONICAL_RUNS)))
assert all(extract_model_key(r) == SCPROTO_KEY for r in SCPROTO_CANONICAL_RUNS), (
    "SCPROTO_CANONICAL_RUNS entries no longer normalize to one shared key -- "
    "extract_model_key's stripping patterns changed; update this cell."
)

MODEL_KEYWORDS = {SCPROTO_KEY: 'scProto'}
MODEL_KEYWORDS['seacell'] = 'SEACells (PCA)'
MODEL_KEYWORDS['metaq'] = 'MetaQ'

# Parametric UMAP -- EXACT folder names per dataset, not a bare 'param_umap' substring
# keyword. Two reasons this has to be exact:
#   1. extract_model_key() strips '_cvae_e\d+' as a "dataset-specific" token, so a
#      cvae_e50 run and a cvae_e100 run normalize to the SAME stripped key -- a
#      substring keyword can't tell them apart and _resolve_run_dir silently picks
#      whichever sorts last alphabetically (was picking cvae_e100 in some cells,
#      cvae_e50 in others -- inconsistent across this same notebook run).
#   2. A bare 'param_umap' keyword never equals any real folder name (every param_umap
#      run has a dataset-specific suffix baked in), so it never matched via
#      _keep_and_rename_runs()'s exact-isin check below -- Parametric UMAP was
#      silently MISSING from Table 1 entirely even though it appeared in Table 2 and
#      the significance tests (which resolve via substring match instead).
# Pinned to each dataset's cvae_e50 run (matching scProto's own Stage-1 epoch budget),
# with no extra bs{n} suffix for pbmc-immune (scProto's own canonical run there also
# has no bs suffix, i.e. default batch size) -- verified these exact names against the
# Drive folder listing directly.
MODEL_KEYWORDS['param_umap_ds-panc_NP220_aff-arbf_cvae_e50_v31'] = 'Parametric UMAP'
MODEL_KEYWORDS['param_umap_ds-lung_aff-arbf_cvae_e50_v31'] = 'Parametric UMAP'
MODEL_KEYWORDS['param_umap_aff-arbf_cvae_e50_v31'] = 'Parametric UMAP'

# One {SEACells, Leiden} entry per correction method (stage1z/harmony/scvi_gauss).
# 'scvi_gauss' is likewise exact-match-only for the same reason as scVI above: a bare
# 'scvi' keyword would substring-match both 'seacell_X_scvi' (plain ZINB) and
# 'seacell_X_scvi_gauss', silently picking whichever sorts last alphabetically (always
# the _gauss one) -- mislabeling ZINB-vs-Gaussian results either way. Using the full
# 'scvi_gauss' method name as the dict key sidesteps this: it can only exactly match
# the Gaussian folder, never the plain one.
# 'harmony' is dimension-qualified ('seacell_X_harmony_d{n}') because
# run_correction_method corrects it at scProto's own latent dim, not the usual 50 --
# handled explicitly below instead of by the generic loop.
for _method in CORRECTION_METHODS:
    if _method == 'harmony':
        continue
    _disp = METHOD_DISPLAY_NAMES[_method]
    MODEL_KEYWORDS[f'seacell_X_{_method}'] = f'SEACells ({_disp})'
    MODEL_KEYWORDS[f'leiden_X_{_method}']  = f'Leiden ({_disp})'

HARMONY_DIM = 8  # scProto's latent_dims for these three datasets -- update if that changes
if 'harmony' in CORRECTION_METHODS:
    MODEL_KEYWORDS[f'seacell_X_harmony_d{HARMONY_DIM}'] = 'SEACells (Harmony)'
    MODEL_KEYWORDS[f'leiden_X_harmony_d{HARMONY_DIM}']  = 'Leiden (Harmony)'

MODEL_KEYWORDS


{'proto_umap_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp': 'scProto',
 'seacell': 'SEACells (PCA)',
 'metaq': 'MetaQ',
 'param_umap_ds-panc_NP220_aff-arbf_cvae_e50_v31': 'Parametric UMAP',
 'param_umap_ds-lung_aff-arbf_cvae_e50_v31': 'Parametric UMAP',
 'param_umap_aff-arbf_cvae_e50_v31': 'Parametric UMAP',
 'seacell_X_stage1z': 'SEACells (scPoli (cVAE))',
 'leiden_X_stage1z': 'Leiden (scPoli (cVAE))',
 'seacell_X_scvi_gauss': 'SEACells (scVI (Gaussian))',
 'leiden_X_scvi_gauss': 'Leiden (scVI (Gaussian))',
 'seacell_X_harmony_d8': 'SEACells (Harmony)',
 'leiden_X_harmony_d8': 'Leiden (Harmony)'}

In [28]:
def _keep_and_rename_runs(df, model_keywords=MODEL_KEYWORDS):
    """Filter load_task1_multi's output down to model_keywords, collapsing each
    method's per-dataset '_K{n}' folder suffix into one shared display row. Drops (with
    a WARNING, not a crash) a stale duplicate run for the same (dataset, display-name)
    pair -- e.g. a leftover leiden_X_harmony_K* folder from an old num_prototypes value.
    """
    runs = df.index.get_level_values('run')
    datasets = df.index.get_level_values('dataset')
    stripped = runs.str.replace(r'_K\d+$', '', regex=True)
    keep_mask = stripped.isin(model_keywords)

    kept_runs, kept_datasets = runs[keep_mask], datasets[keep_mask]
    kept_display = stripped[keep_mask].map(model_keywords)

    out = df[keep_mask].copy()
    out.index = pd.MultiIndex.from_arrays([kept_datasets, kept_display], names=['dataset', 'run'])

    dupe_mask = out.index.duplicated(keep='first')
    if dupe_mask.any():
        stale = list(zip(kept_datasets[dupe_mask], kept_runs[dupe_mask], kept_display[dupe_mask]))
        print(f"WARNING: dropped {dupe_mask.sum()} duplicate (dataset, method) row(s) -- "
              f"likely a stale run at an old K value still on disk. "
              f"(dataset, on-disk folder, display name): {stale}")
        out = out[~dupe_mask]
    return out


## Table 1 (headline table): modularity, batch entropy, purity

In [29]:
df_task1 = load_task1_multi(RNA_SEQ_DATASETS, metrics=TASK1_METRICS)
df_task1 = _keep_and_rename_runs(df_task1)
show_table(df_task1, metrics=TASK1_METRICS, dataset_display_names=dataset_display_names)


## Table 2 (rare-cell metrics): coverage, homogeneity, F1

In [30]:
df_rare = rare_celltype_purity_table(RNA_SEQ_DATASETS, model_keywords=MODEL_KEYWORDS, verbose=True)

_dupe_mask = df_rare.index.duplicated(keep='first')
if _dupe_mask.any():
    print(f"WARNING: dropped {_dupe_mask.sum()} duplicate row(s) from df_rare: "
          f"{df_rare.index[_dupe_mask].tolist()}")
    df_rare = df_rare[~_dupe_mask]

show_table(
    df_rare,
    metrics=[
        'batch_rare_coverage_mean', 'batch_rare_recall_macro_mean',
        'batch_rare_precision_macro_mean', 'batch_rare_homogeneity_mean',
        'batch_rare_cross_batch_homog_mean', 'batch_rare_f1_macro_mean',
    ],
    dataset_display_names=dataset_display_names,
)


  [scProto|pancreas] resolving run dir ...
  [SEACells (PCA)|pancreas] resolving run dir ...
  [MetaQ|pancreas] resolving run dir ...
  [Parametric UMAP|pancreas] resolving run dir ...
  [Parametric UMAP|pancreas] resolving run dir ...
  [Parametric UMAP|pancreas] resolving run dir ...  [SEACells (PCA)|pancreas] run dir resolved (0.0s)
  [SEACells (scPoli (cVAE))|pancreas] resolving run dir ...

  [Leiden (scPoli (cVAE))|pancreas] resolving run dir ...
  [SEACells (PCA)|pancreas] reading umap_cells.csv ...
  [SEACells (PCA)|pancreas] umap_cells.csv loaded (16382 rows, 0.0s)
  [SEACells (PCA)|pancreas] reading umap_protos.csv ...
  [SEACells (PCA)|pancreas] umap_protos.csv loaded (0.0s)
  [SEACells (PCA)|pancreas] batch='tech' | 9 unique values, e.g. ['celseq', 'celseq2', 'fluidigmc1', 'inDrop1', 'inDrop2']
  [SEACells (PCA)|pancreas] label='celltype', batch='tech' | computing metacell label fractions ...
  [SEACells (PCA)|pancreas] label fractions done (0.0s)
  [Parametric UMAP|pancrea

## Significance tests

Reviewer 3 (e9Ho), Weakness 7: *"Tables 1-2 report no significance test ... some 'wins'
overlap in std."* Everything below tests scProto against every same-K baseline
(realized metacell count within 5% of scProto's -- a baseline whose K genuinely differs
is dropped from that comparison rather than tested, per the paper's own "same K as
scProto" baseline protocol) at alpha=0.05, Bonferroni-corrected across baselines within
each (dataset, metric) group.


### Table 1 -- unpaired one-sided Mann-Whitney U (scProto > baseline)

Runs directly on the saved `modularity_per_batch.csv` / `purity_per_mc.csv` /
`batch_entropy_per_mc.csv` in each run's directory -- no retraining needed.


In [31]:
sig_df_table1 = graph_batch_significance(
    RNA_SEQ_DATASETS,
    MODEL_KEYWORDS,
    ref_name='scProto',
    dataset_display_names=dataset_display_names,
)

for metric_name in sig_df_table1['metric'].unique():
    print(f"=== {metric_name}: scProto vs. each same-K baseline, one-sided "
          f"Mann-Whitney U (scProto > other), Bonferroni-corrected per dataset ===")
    sub = sig_df_table1[sig_df_table1['metric'] == metric_name].copy()
    sub['cell'] = sub.apply(
        lambda r: f"{r['median']:.3f} (K={r['k']}, n={r['n']}) [ref]" if r['method'] == 'scProto'
        else f"{r['median']:.3f} (K={r['k']}, n={r['n']})  {r.get('sig', '?')}  p_adj={r.get('p_adj', float('nan')):.3g}",
        axis=1,
    )
    display(sub.pivot(index='method', columns='dataset', values='cell'))

sig_df_table1


  [fig] 'leiden_X_stage1z' matched 2 runs — using 'leiden_X_stage1z_K88'
Skipped (K mismatch vs reference -- not a same-K comparison, per the paper's baseline protocol):
  - Immune/modularity_per_batch: Leiden (scPoli (cVAE)) skipped (K=88 vs scProto's K=294, outside 5% tolerance)
  - Immune/purity_per_mc: Leiden (scPoli (cVAE)) skipped (K=88 vs scProto's K=294, outside 5% tolerance)
  - Immune/batch_entropy_per_mc: Leiden (scPoli (cVAE)) skipped (K=88 vs scProto's K=294, outside 5% tolerance)
=== modularity_per_batch: scProto vs. each same-K baseline, one-sided Mann-Whitney U (scProto > other), Bonferroni-corrected per dataset ===


dataset,Immune,Lung,Pancreas
method,,,
Leiden (Harmony),"0.250 (K=300, n=5) * p_adj=0.0317","0.403 (K=300, n=16) *** p_adj=8.39e-06","0.614 (K=220, n=9) ns p_adj=1"
Leiden (scPoli (cVAE)),NaN,"0.702 (K=300, n=16) ns p_adj=1","0.612 (K=220, n=9) ns p_adj=1"
Leiden (scVI (Gaussian)),"0.344 (K=300, n=5) ns p_adj=0.127","0.300 (K=300, n=16) *** p_adj=6.95e-06","0.340 (K=220, n=9) ** p_adj=0.00489"
MetaQ,"0.284 (K=294, n=5) * p_adj=0.0317","0.403 (K=292, n=16) *** p_adj=0.000119","0.391 (K=217, n=9) * p_adj=0.0161"
Parametric UMAP,"0.233 (K=300, n=5) * p_adj=0.0317","0.312 (K=300, n=16) *** p_adj=6.95e-06","0.242 (K=220, n=9) ** p_adj=0.00186"
SEACells (Harmony),"0.551 (K=300, n=5) ns p_adj=0.127","0.625 (K=300, n=16) ** p_adj=0.00144","0.566 (K=220, n=9) ns p_adj=0.287"
SEACells (PCA),"0.569 (K=300, n=5) ns p_adj=0.222","0.671 (K=300, n=16) ns p_adj=1","0.658 (K=220, n=9) ns p_adj=1"
SEACells (scPoli (cVAE)),"0.597 (K=300, n=5) ns p_adj=1","0.632 (K=300, n=16) ** p_adj=0.00572","0.546 (K=220, n=9) ns p_adj=0.42"
SEACells (scVI (Gaussian)),"0.657 (K=300, n=5) ns p_adj=1","0.141 (K=300, n=16) *** p_adj=6.95e-06","0.720 (K=220, n=9) ns p_adj=1"


=== purity_per_mc: scProto vs. each same-K baseline, one-sided Mann-Whitney U (scProto > other), Bonferroni-corrected per dataset ===


dataset,Immune,Lung,Pancreas
method,,,
Leiden (Harmony),"0.855 (K=300, n=300) *** p_adj=4.13e-17","0.841 (K=300, n=300) *** p_adj=2.09e-12","1.000 (K=220, n=220) ns p_adj=1"
Leiden (scPoli (cVAE)),NaN,"0.962 (K=300, n=300) ns p_adj=0.0722","1.000 (K=220, n=220) ns p_adj=1"
Leiden (scVI (Gaussian)),"0.799 (K=300, n=300) *** p_adj=1.43e-34","0.831 (K=300, n=300) *** p_adj=1.2e-11","1.000 (K=220, n=220) ns p_adj=1"
MetaQ,"0.842 (K=294, n=294) *** p_adj=1.61e-27","0.974 (K=292, n=292) ns p_adj=1","0.991 (K=217, n=217) ns p_adj=1"
Parametric UMAP,"0.857 (K=300, n=300) *** p_adj=5.13e-26","0.894 (K=300, n=300) *** p_adj=3.34e-10","0.989 (K=220, n=220) ns p_adj=1"
SEACells (Harmony),"0.883 (K=300, n=300) *** p_adj=7.05e-15","0.832 (K=300, n=300) *** p_adj=1.44e-11","0.967 (K=220, n=220) *** p_adj=0.000334"
SEACells (PCA),"0.965 (K=300, n=300) *** p_adj=1.44e-05","0.977 (K=300, n=300) ns p_adj=1","0.993 (K=220, n=220) ns p_adj=1"
SEACells (scPoli (cVAE)),"0.951 (K=300, n=300) *** p_adj=3.05e-08","0.946 (K=300, n=300) ** p_adj=0.00843","0.989 (K=220, n=220) ns p_adj=1"
SEACells (scVI (Gaussian)),"0.906 (K=300, n=300) *** p_adj=6.93e-15","0.944 (K=300, n=300) *** p_adj=0.000286","0.981 (K=220, n=220) ns p_adj=0.118"


=== batch_entropy_per_mc: scProto vs. each same-K baseline, one-sided Mann-Whitney U (scProto > other), Bonferroni-corrected per dataset ===


dataset,Immune,Lung,Pancreas
method,,,
Leiden (Harmony),"1.071 (K=300, n=300) ns p_adj=1","1.350 (K=300, n=300) ns p_adj=1","1.327 (K=220, n=220) ns p_adj=1"
Leiden (scPoli (cVAE)),NaN,"1.262 (K=300, n=300) ns p_adj=1","1.306 (K=220, n=220) ns p_adj=1"
Leiden (scVI (Gaussian)),"0.769 (K=300, n=300) ns p_adj=1","1.122 (K=300, n=300) ns p_adj=1","0.805 (K=220, n=220) ns p_adj=1"
MetaQ,"0.426 (K=294, n=294) ns p_adj=1","0.336 (K=292, n=292) ns p_adj=1","0.245 (K=217, n=217) ns p_adj=1"
Parametric UMAP,"1.125 (K=300, n=300) ns p_adj=1","1.423 (K=300, n=300) ns p_adj=1","1.344 (K=220, n=220) ns p_adj=1"
SEACells (Harmony),"0.726 (K=300, n=300) ns p_adj=1","1.526 (K=300, n=300) ns p_adj=1","1.339 (K=220, n=220) ns p_adj=1"
SEACells (PCA),"-0.000 (K=300, n=300) ns p_adj=1","0.218 (K=300, n=300) ns p_adj=1","-0.000 (K=220, n=220) *** p_adj=1.44e-05"
SEACells (scPoli (cVAE)),"0.515 (K=300, n=300) ns p_adj=1","1.335 (K=300, n=300) ns p_adj=1","1.261 (K=220, n=220) ns p_adj=1"
SEACells (scVI (Gaussian)),"0.434 (K=300, n=300) ns p_adj=1","1.116 (K=300, n=300) ns p_adj=1","0.882 (K=220, n=220) ns p_adj=1"


,dataset,metric,method,k,n,median,mean,std,p_vs_ref,p_adj,sig
0,Pancreas,modularity_per_batch,scProto,219,9,0.621456,0.601234,0.083385,NaN,NaN,NaN
1,Pancreas,modularity_per_batch,SEACells (PCA),220,9,0.657892,0.673906,0.050910,0.973971,1.000000,ns
2,Pancreas,modularity_per_batch,MetaQ,217,9,0.390540,0.408372,0.092613,0.001784,0.016059,*
3,Pancreas,modularity_per_batch,Parametric UMAP,220,9,0.241827,0.252182,0.061234,0.000206,0.001855,**
4,Pancreas,modularity_per_batch,SEACells (scPoli (cVAE)),220,9,0.545776,0.555830,0.026080,0.046699,0.420290,ns
...,...,...,...,...,...,...,...,...,...,...,...
82,Immune,batch_entropy_per_mc,SEACells (scPoli (cVAE)),300,300,0.514597,0.540357,0.481987,1.000000,1.000000,ns
83,Immune,batch_entropy_per_mc,SEACells (scVI (Gaussian)),300,300,0.434307,0.496806,0.475820,1.000000,1.000000,ns
84,Immune,batch_entropy_per_mc,Leiden (scVI (Gaussian)),300,300,0.769386,0.710373,0.406745,1.000000,1.000000,ns
85,Immune,batch_entropy_per_mc,SEACells (Harmony),300,300,0.725965,0.640662,0.511225,1.000000,1.000000,ns


### Table 1 -- paired one-sided Wilcoxon signed-rank on modularity (recommended)

Only `modularity_per_batch` is pairable: batch is a shared, method-independent unit (the
same physical batch exists for every method on a dataset), so `ref_vals[i]` and
`vals[i]` refer to the same batch. Pairing removes shared batch-to-batch noise and is
more powerful than the unpaired test above on this exact data -- no new runs needed.
`purity_per_mc` / `batch_entropy_per_mc` stay unpaired (a metacell is method-specific,
so there's no natural cross-method correspondence to pair on -- that's the statistically
correct call, not a limitation).


In [32]:
sig_df_table1_paired = graph_batch_significance_paired(
    RNA_SEQ_DATASETS,
    MODEL_KEYWORDS,
    ref_name='scProto',
    dataset_display_names=dataset_display_names,
)

sub = sig_df_table1_paired.copy()
sub['cell'] = sub.apply(
    lambda r: f"{r['median']:.3f} (K={r['k']}, n={r['n']}) [ref]" if r['method'] == 'scProto'
    else f"{r['median']:.3f} (K={r['k']}, n={r['n']}, wins={r.get('n_wins', '?')}/{r['n']})  "
         f"{r.get('sig', '?')}  p_adj={r.get('p_adj', float('nan')):.3g}",
    axis=1,
)
display(sub.pivot(index='method', columns='dataset', values='cell'))

sig_df_table1_paired


  [fig] 'leiden_X_stage1z' matched 2 runs — using 'leiden_X_stage1z_K88'
Skipped (K mismatch vs reference -- not a same-K comparison, per the paper's baseline protocol):
  - Immune/modularity_per_batch: Leiden (scPoli (cVAE)) skipped (K=88 vs scProto's K=294, outside 5% tolerance)


dataset,Immune,Lung,Pancreas
method,,,
Leiden (Harmony),"0.250 (K=300, n=5, wins=5.0/5) ns p_adj=0.25","0.403 (K=300, n=16, wins=16.0/16) *** p_adj=...","0.614 (K=220, n=9, wins=6.0/9) ns p_adj=1"
Leiden (scPoli (cVAE)),NaN,"0.702 (K=300, n=16, wins=2.0/16) ns p_adj=1","0.612 (K=220, n=9, wins=4.0/9) ns p_adj=1"
Leiden (scVI (Gaussian)),"0.344 (K=300, n=5, wins=4.0/5) ns p_adj=0.5","0.300 (K=300, n=16, wins=16.0/16) *** p_adj=...","0.340 (K=220, n=9, wins=8.0/9) * p_adj=0.0352"
MetaQ,"0.284 (K=294, n=5, wins=5.0/5) ns p_adj=0.25","0.403 (K=292, n=16, wins=15.0/16) *** p_adj=...","0.391 (K=217, n=9, wins=8.0/9) * p_adj=0.0352"
Parametric UMAP,"0.233 (K=300, n=5, wins=5.0/5) ns p_adj=0.25","0.312 (K=300, n=16, wins=16.0/16) *** p_adj=...","0.242 (K=220, n=9, wins=9.0/9) * p_adj=0.0176"
SEACells (Harmony),"0.551 (K=300, n=5, wins=5.0/5) ns p_adj=0.25","0.625 (K=300, n=16, wins=14.0/16) ** p_adj=0...","0.566 (K=220, n=9, wins=7.0/9) ns p_adj=1"
SEACells (PCA),"0.569 (K=300, n=5, wins=5.0/5) ns p_adj=0.25","0.671 (K=300, n=16, wins=9.0/16) ns p_adj=1","0.658 (K=220, n=9, wins=0.0/9) ns p_adj=1"
SEACells (scPoli (cVAE)),"0.597 (K=300, n=5, wins=3.0/5) ns p_adj=1","0.632 (K=300, n=16, wins=14.0/16) ** p_adj=0...","0.546 (K=220, n=9, wins=7.0/9) ns p_adj=0.914"
SEACells (scVI (Gaussian)),"0.657 (K=300, n=5, wins=1.0/5) ns p_adj=1","0.141 (K=300, n=16, wins=16.0/16) *** p_adj=...","0.720 (K=220, n=9, wins=0.0/9) ns p_adj=1"


,dataset,metric,method,k,n,median,mean,std,n_wins,p_vs_ref,p_adj,sig
0,Pancreas,modularity_per_batch,scProto,219,9,0.621456,0.601234,0.083385,NaN,NaN,NaN,NaN
1,Pancreas,modularity_per_batch,SEACells (PCA),220,9,0.657892,0.673906,0.050910,0.0,1.000000,1.000000,ns
2,Pancreas,modularity_per_batch,MetaQ,217,9,0.390540,0.408372,0.092613,8.0,0.003906,0.035156,*
3,Pancreas,modularity_per_batch,Parametric UMAP,220,9,0.241827,0.252182,0.061234,9.0,0.001953,0.017578,*
4,Pancreas,modularity_per_batch,SEACells (scPoli (cVAE)),220,9,0.545776,0.555830,0.026080,7.0,0.101562,0.914062,ns
5,Pancreas,modularity_per_batch,Leiden (scPoli (cVAE)),220,9,0.611845,0.621367,0.026665,4.0,0.820312,1.000000,ns
6,Pancreas,modularity_per_batch,SEACells (scVI (Gaussian)),220,9,0.720242,0.725213,0.038824,0.0,1.000000,1.000000,ns
7,Pancreas,modularity_per_batch,Leiden (scVI (Gaussian)),220,9,0.340486,0.352439,0.100534,8.0,0.003906,0.035156,*
8,Pancreas,modularity_per_batch,SEACells (Harmony),220,9,0.566367,0.567150,0.017709,7.0,0.125000,1.000000,ns
9,Pancreas,modularity_per_batch,Leiden (Harmony),220,9,0.614318,0.612469,0.021231,6.0,0.544922,1.000000,ns


### Table 2 (bonus) -- paired one-sided Wilcoxon signed-rank on rare-cell F1/homogeneity

Same reviewer weakness explicitly names Table 2 too ("Tables 1-2 report no significance
test"). Runs on `df_rare`'s raw per-batch arrays computed above.


In [33]:
df_sig_paired = rare_metric_significance_paired(
    df_rare,
    ref_name='scProto',
    metrics=(
        '_batch_rare_f1_macro_per_batch',
        '_batch_rare_homogeneity_per_batch',
        '_batch_rare_cross_batch_homog_per_batch',
    ),
    dataset_display_names=dataset_display_names,
)

for metric_name in df_sig_paired['metric'].unique():
    print(f"=== {metric_name}: scProto vs. each same-K baseline, PAIRED one-sided "
          f"Wilcoxon signed-rank (scProto > other), Bonferroni-corrected per dataset ===")
    sub = df_sig_paired[df_sig_paired['metric'] == metric_name].copy()
    sub['cell'] = sub.apply(
        lambda r: f"{r['median']:.3f} (K={r['k']}, n={r['n']}) [ref]" if r['method'] == 'scProto'
        else f"{r['median']:.3f} (K={r['k']}, n={r['n']}, wins={r.get('n_wins', '?')}/{r['n']})  "
             f"{r.get('sig', '?')}  p_adj={r.get('p_adj', float('nan')):.3g}",
        axis=1,
    )
    display(sub.pivot(index='method', columns='dataset', values='cell'))

df_sig_paired


Skipped (K mismatch vs reference -- not a same-K comparison, per the paper's baseline protocol):
  - Immune/batch_rare_f1_macro: Leiden (scPoli (cVAE)) skipped (K=88 vs scProto's K=294, outside 5% tolerance)
  - Immune/batch_rare_homogeneity: Leiden (scPoli (cVAE)) skipped (K=88 vs scProto's K=294, outside 5% tolerance)
  - Immune/batch_rare_cross_batch_homog: Leiden (scPoli (cVAE)) skipped (K=88 vs scProto's K=294, outside 5% tolerance)
=== batch_rare_f1_macro: scProto vs. each same-K baseline, PAIRED one-sided Wilcoxon signed-rank (scProto > other), Bonferroni-corrected per dataset ===


dataset,Immune,Lung,Pancreas
method,,,
Leiden (Harmony),"0.766 (K=300, n=5, wins=4.0/5) ns p_adj=0.5","0.282 (K=300, n=15, wins=13.0/15) ** p_adj=0...","0.133 (K=220, n=8, wins=8.0/8) * p_adj=0.0352"
Leiden (scPoli (cVAE)),NaN,"0.565 (K=300, n=15, wins=11.0/15) ns p_adj=1","0.526 (K=220, n=8, wins=5.0/8) ns p_adj=1"
Leiden (scVI (Gaussian)),"0.676 (K=300, n=5, wins=5.0/5) ns p_adj=0.25","0.299 (K=300, n=15, wins=15.0/15) *** p_adj=...","0.000 (K=220, n=8, wins=8.0/8) * p_adj=0.0352"
MetaQ,"0.796 (K=294, n=5, wins=4.0/5) ns p_adj=1","0.627 (K=292, n=15, wins=7.0/15) ns p_adj=1","0.069 (K=217, n=8, wins=8.0/8) * p_adj=0.0352"
Parametric UMAP,"0.746 (K=300, n=5, wins=4.0/5) ns p_adj=0.5","0.562 (K=300, n=15, wins=10.0/15) ns p_adj=1","0.349 (K=220, n=8, wins=6.0/8) ns p_adj=0.352"
SEACells (Harmony),"0.812 (K=300, n=5, wins=4.0/5) ns p_adj=0.5","0.348 (K=300, n=15, wins=13.0/15) ** p_adj=0...","0.296 (K=220, n=8, wins=7.0/8) ns p_adj=0.105"
SEACells (PCA),"0.923 (K=300, n=5, wins=2.0/5) ns p_adj=1","0.498 (K=300, n=15, wins=11.0/15) * p_adj=0....","0.373 (K=220, n=8, wins=6.0/8) ns p_adj=0.352"
SEACells (scPoli (cVAE)),"0.883 (K=300, n=5, wins=3.0/5) ns p_adj=1","0.551 (K=300, n=15, wins=10.0/15) ns p_adj=1","0.322 (K=220, n=8, wins=6.0/8) ns p_adj=0.246"
SEACells (scVI (Gaussian)),"0.652 (K=300, n=5, wins=5.0/5) ns p_adj=0.25","0.342 (K=300, n=15, wins=15.0/15) *** p_adj=...","0.080 (K=220, n=8, wins=7.0/8) ns p_adj=0.0703"


=== batch_rare_homogeneity: scProto vs. each same-K baseline, PAIRED one-sided Wilcoxon signed-rank (scProto > other), Bonferroni-corrected per dataset ===


dataset,Immune,Lung,Pancreas
method,,,
Leiden (Harmony),"0.707 (K=300, n=5, wins=5.0/5) ns p_adj=0.25","0.345 (K=300, n=15, wins=13.0/15) ** p_adj=0...","0.220 (K=220, n=8, wins=8.0/8) * p_adj=0.0352"
Leiden (scPoli (cVAE)),NaN,"0.508 (K=300, n=15, wins=12.0/15) ns p_adj=0...","0.440 (K=220, n=8, wins=7.0/8) ns p_adj=0.879"
Leiden (scVI (Gaussian)),"0.538 (K=300, n=5, wins=5.0/5) ns p_adj=0.25","0.327 (K=300, n=15, wins=15.0/15) *** p_adj=...","0.119 (K=220, n=8, wins=7.0/8) ns p_adj=0.0703"
MetaQ,"0.660 (K=294, n=5, wins=4.0/5) ns p_adj=0.5","0.510 (K=292, n=15, wins=9.0/15) ns p_adj=1","0.174 (K=217, n=8, wins=8.0/8) * p_adj=0.0352"
Parametric UMAP,"0.627 (K=300, n=5, wins=5.0/5) ns p_adj=0.25","0.570 (K=300, n=15, wins=10.0/15) ns p_adj=1","0.401 (K=220, n=8, wins=7.0/8) ns p_adj=0.246"
SEACells (Harmony),"0.804 (K=300, n=5, wins=5.0/5) ns p_adj=0.25","0.408 (K=300, n=15, wins=14.0/15) * p_adj=0....","0.269 (K=220, n=8, wins=8.0/8) * p_adj=0.0352"
SEACells (PCA),"0.815 (K=300, n=5, wins=3.0/5) ns p_adj=1","0.495 (K=300, n=15, wins=13.0/15) * p_adj=0....","0.369 (K=220, n=8, wins=8.0/8) * p_adj=0.0352"
SEACells (scPoli (cVAE)),"0.862 (K=300, n=5, wins=1.0/5) ns p_adj=1","0.507 (K=300, n=15, wins=10.0/15) ns p_adj=0...","0.363 (K=220, n=8, wins=7.0/8) ns p_adj=0.0703"
SEACells (scVI (Gaussian)),"0.582 (K=300, n=5, wins=5.0/5) ns p_adj=0.25","0.385 (K=300, n=15, wins=14.0/15) *** p_adj=...","0.074 (K=220, n=8, wins=7.0/8) ns p_adj=0.0703"


=== batch_rare_cross_batch_homog: scProto vs. each same-K baseline, PAIRED one-sided Wilcoxon signed-rank (scProto > other), Bonferroni-corrected per dataset ===


dataset,Immune,Lung,Pancreas
method,,,
Leiden (Harmony),"0.373 (K=300, n=5, wins=2.0/5) ns p_adj=1","0.341 (K=300, n=15, wins=13.0/15) * p_adj=0....","0.184 (K=220, n=8, wins=6.0/8) ns p_adj=0.352"
Leiden (scPoli (cVAE)),NaN,"0.478 (K=300, n=15, wins=10.0/15) ns p_adj=1","0.369 (K=220, n=8, wins=1.0/8) ns p_adj=1"
Leiden (scVI (Gaussian)),"0.344 (K=300, n=5, wins=4.0/5) ns p_adj=0.272","0.318 (K=300, n=15, wins=14.0/15) *** p_adj=...","0.064 (K=220, n=8, wins=8.0/8) * p_adj=0.0352"
MetaQ,"0.465 (K=294, n=5, wins=2.0/5) ns p_adj=1","0.450 (K=292, n=15, wins=7.0/15) ns p_adj=1","0.107 (K=217, n=8, wins=7.0/8) ns p_adj=0.0703"
Parametric UMAP,"0.455 (K=300, n=5, wins=0.0/5) ns p_adj=1","0.549 (K=300, n=15, wins=8.0/15) ns p_adj=1","0.346 (K=220, n=8, wins=4.0/8) ns p_adj=1"
SEACells (Harmony),"0.305 (K=300, n=5, wins=3.0/5) ns p_adj=1","0.399 (K=300, n=15, wins=14.0/15) * p_adj=0....","0.210 (K=220, n=8, wins=6.0/8) ns p_adj=0.668"
SEACells (PCA),"0.029 (K=300, n=5, wins=3.0/5) ns p_adj=0.577","0.416 (K=300, n=15, wins=12.0/15) * p_adj=0....","0.212 (K=220, n=8, wins=5.0/8) ns p_adj=1"
SEACells (scPoli (cVAE)),"0.443 (K=300, n=5, wins=2.0/5) ns p_adj=1","0.470 (K=300, n=15, wins=7.0/15) ns p_adj=1","0.266 (K=220, n=8, wins=5.0/8) ns p_adj=1"
SEACells (scVI (Gaussian)),"0.347 (K=300, n=5, wins=4.0/5) ns p_adj=0.272","0.377 (K=300, n=15, wins=13.0/15) ** p_adj=0...","0.041 (K=220, n=8, wins=8.0/8) * p_adj=0.0352"


,dataset,metric,method,k,n,median,mean,std,n_wins,p_vs_ref,p_adj,sig
0,Pancreas,batch_rare_f1_macro,scProto,219,8,0.437680,0.535403,0.207961,NaN,NaN,NaN,NaN
1,Pancreas,batch_rare_f1_macro,SEACells (PCA),220,8,0.372628,0.370589,0.250869,6.0,0.039062,0.351562,ns
2,Pancreas,batch_rare_f1_macro,MetaQ,217,8,0.068861,0.109538,0.136172,8.0,0.003906,0.035156,*
3,Pancreas,batch_rare_f1_macro,Parametric UMAP,220,8,0.348830,0.410065,0.228790,6.0,0.039062,0.351562,ns
4,Pancreas,batch_rare_f1_macro,SEACells (scPoli (cVAE)),220,8,0.322173,0.398344,0.229711,6.0,0.027344,0.246094,ns
...,...,...,...,...,...,...,...,...,...,...,...,...
82,Immune,batch_rare_cross_batch_homog,SEACells (scPoli (cVAE)),300,5,0.442893,0.392581,0.204611,2.0,0.781250,1.000000,ns
83,Immune,batch_rare_cross_batch_homog,SEACells (scVI (Gaussian)),300,5,0.347206,0.273950,0.149657,4.0,0.033945,0.271557,ns
84,Immune,batch_rare_cross_batch_homog,Leiden (scVI (Gaussian)),300,5,0.344439,0.275207,0.152992,4.0,0.033945,0.271557,ns
85,Immune,batch_rare_cross_batch_homog,SEACells (Harmony),300,5,0.304594,0.245410,0.153361,3.0,0.156250,1.000000,ns


## Reading the results for the rebuttal

- `sig` column: `***`/`**`/`*` = significant at p_adj < 0.001/0.01/0.05 after Bonferroni
  correction; `ns` = not significant at alpha=0.05 -- an honest result to report, not a
  bug (e.g. the Immune modularity 0.62+-0.08 vs SEACells' 0.55+-0.04 case the reviewer
  flagged is exactly the kind of comparison this is meant to settle either way).
- `n_wins` (paired tests only): out of `n` batches, how many scProto's value beats the
  baseline's outright -- readable even when `p_adj` stays `ns`.
- Any "Skipped (K mismatch ...)" printout above means that baseline's realized metacell
  count didn't match scProto's within 5% for that dataset, so it wasn't tested at all
  (per the paper's "same K as scProto" protocol) -- not a bug, and not a result either.
- If `Parametric UMAP` or `MetaQ` don't appear for some dataset, check the printed
  "[fig] 'x' matched N runs" / missing-keyword messages above and the actual folder
  names under `MODEL_DIR/<ds_id>/` -- the keywords here are plain substring matches, not
  hardcoded exact folder names like scProto's.
